> **LEGACY.** Reconstruction-only runner from an earlier version (GPU, `botrecon.run` only). Use `reproduce_kaggle_cpu.ipynb`, which runs the complete protocol on CPU. Kept for reference; not maintained.

# Independent reconstruction and reproduction: hybrid metadata + text + GCN + LightGBM bot detector

**Read first.** The original implementation of the manuscript is lost. This notebook runs a *reconstruction* specified in the
Reconstruction Specification document. Every result row it writes carries a provenance label:

* `REPRODUCED-CORRECTED`: main results for the revised paper
* `REPRODUCED-LITERAL`: manuscript-faithful protocol, sensitivity analysis only
* `DIAGNOSTIC-INVALID`: deliberately flawed run (early stopping on test); never report as a result
* `SMOKE-TEST-NOT-A-RESULT`: installation check

Manuscript-reported numbers (98.57% / 92.28%) are **not** produced here; they appear only in their own labelled rows.

Runtime: GPU (Runtime → Change runtime type → T4 GPU).

In [ ]:
# 1. Dependencies (PyTorch is preinstalled on Colab and Kaggle)
!pip -q install lightgbm transformers sentence-transformers
import torch, lightgbm, transformers, sentence_transformers
print(torch.__version__, torch.cuda.is_available(), lightgbm.__version__, transformers.__version__, sentence_transformers.__version__)

In [ ]:
# 2. Code: upload botrecon_repo.zip (Files panel) and unzip
!unzip -q -o botrecon_repo.zip -d /content && ls /content/botrecon_repo
%cd /content/botrecon_repo

In [ ]:
# 3. Tests. All must pass, including the 3 real-backend tests that were skipped where the code was written.
!python -m pytest -q -rs

## 4. Data

**Cresci-2017**: download the official release from the Botometer bot repository (manuscript ref. [27]).
The archive may contain nested zips; the cell below extracts recursively and finds the folder holding `genuine_accounts.csv`.

**TwiBot-20**: obtained from the dataset authors on request. Put `train.json`, `dev.json`, `test.json` in a Drive folder.

In [ ]:
import os, zipfile, glob
CRESCI_ZIP = "/content/cresci-2017.csv.zip"          # upload or wget the official file here
# !wget -O /content/cresci-2017.csv.zip "https://botometer.osome.iu.edu/bot-repository/datasets/cresci-2017/cresci-2017.csv.zip"
def extract_all(path, dest):
    with zipfile.ZipFile(path) as z: z.extractall(dest)
    done = set()
    while True:
        todo = [z for z in glob.glob(os.path.join(dest, "**", "*.zip"), recursive=True) if z not in done]
        if not todo: break
        for zp in todo:
            with zipfile.ZipFile(zp) as z: z.extractall(os.path.dirname(zp))
            done.add(zp)
if os.path.exists(CRESCI_ZIP):
    extract_all(CRESCI_ZIP, "/content/cresci")
    hits = [os.path.dirname(p) for p in glob.glob("/content/cresci/**/genuine_accounts.csv*", recursive=True)]
    CRESCI_ROOT = sorted(set(hits))[0]; print("CRESCI_ROOT =", CRESCI_ROOT); print(sorted(os.listdir(CRESCI_ROOT)))

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
TWIBOT_ROOT = "/content/drive/MyDrive/TwiBot-20"     # folder with train.json, dev.json, test.json
print(os.listdir(TWIBOT_ROOT))

In [ ]:
# 5. Data audit (no model): counts per group, labels, dedup, empty text. Compare with published group sizes.
from botrecon.data import load_cresci2017, load_twibot20
from botrecon.text import text_inputs, empty_rates
import json
c = load_cresci2017(CRESCI_ROOT, "explicit", load_tweets=True)
print(json.dumps(c.log, indent=1))
print("Cresci empty tweets:", empty_rates(text_inputs(c, "A"), c.y), " empty bios:", empty_rates(text_inputs(c, "B"), c.y))
t = load_twibot20(TWIBOT_ROOT, load_tweets=True)
print(json.dumps(t.log, indent=1))
print("TwiBot empty tweets:", empty_rates(text_inputs(t, "A"), t.y), " empty bios:", empty_rates(text_inputs(t, "B"), t.y))

In [ ]:
# 6. Smoke run on real data with the hashing encoder (installation check; rows labelled SMOKE-TEST)
!python -m botrecon.run --dataset twibot20 --data-root "$TWIBOT_ROOT" --family corrected --seeds 0 --only fusion=P3,tap=hidden --smoke --out results/smoke.csv

In [ ]:
# 7. Corrected family: 8 configs x 5 seeds per dataset (main results)
!python -m botrecon.run --dataset twibot20   --data-root "$TWIBOT_ROOT" --family corrected --seeds 0 1 2 3 4
!python -m botrecon.run --dataset cresci2017 --data-root "$CRESCI_ROOT" --family corrected --seeds 0 1 2 3 4

In [ ]:
# 8. Literal family screen: 1 seed (TwiBot 16 configs, Cresci 32 configs)
!python -m botrecon.run --dataset twibot20   --data-root "$TWIBOT_ROOT" --family literal --seeds 0
!python -m botrecon.run --dataset cresci2017 --data-root "$CRESCI_ROOT" --family literal --seeds 0

In [ ]:
# 9. Primary configuration = best mean VALIDATION F1 among corrected configs (test set not used)
import pandas as pd
from botrecon.report import summarise, primary
s = summarise(pd.read_csv("results/runs.csv"))
for ds in ("twibot20", "cresci2017"):
    p = primary(s, ds); print(ds, "primary:", p.fusion, p.tap, p.text, "val F1", round(p.val_f1_bot_mean, 4))

In [ ]:
# 10. Inductive runs for the primary config, and literal configs closest to the reported numbers at 5 seeds.
# Fill the filters from cell 9 and from table R3 after running the report once.
PRIMARY_TW = "fusion=P3,tap=hidden,text=B"   # <- replace with cell 9 output
PRIMARY_CR = "fusion=P3,tap=hidden,text=B"   # <- replace with cell 9 output
!python -m botrecon.run --dataset twibot20   --data-root "$TWIBOT_ROOT" --family corrected --inductive --seeds 0 1 2 3 4 --only "$PRIMARY_TW"
!python -m botrecon.run --dataset cresci2017 --data-root "$CRESCI_ROOT" --family corrected --inductive --seeds 0 1 2 3 4 --only "$PRIMARY_CR"

In [ ]:
# 11. Diagnostic (INVALID by design): literal closest config with LightGBM early-stopped on TEST.
CLOSEST_TW = "fusion=P3,tap=prob,text=B,twibot_split=random"   # <- replace with R3 'Closest config'
!python -m botrecon.run --dataset twibot20 --data-root "$TWIBOT_ROOT" --family literal --diagnostic --seeds 0 1 2 3 4 --only "$CLOSEST_TW"

In [ ]:
# 12. Tables R1-R6 (Markdown + printed). Download results/ when done; it holds every run and prediction file.
!python -m botrecon.report --runs results/runs.csv --out results/tables.md
!zip -qr /content/results.zip results && echo saved /content/results.zip